# Data Lake vs Delta Lake

---
# O que é um Data Lake?

Um **Data Lake** é um repositório de dados em larga escala capaz de armazenar informações **estruturadas, semiestruturadas e não estruturadas** em seu formato bruto.

Ele é construído sobre armazenamento barato e escalável, como:
- **AWS S3**
- **Azure Data Lake Storage (ADLS Gen2)**
- **Google Cloud Storage (GCS)**

## 📌 Características do Data Lake

✅ Armazena qualquer tipo de dado (texto, JSON, CSV, Parquet, imagens, logs, áudio etc.)  
✅ Baixo custo e alta escalabilidade  
✅ Flexível, permite ingestão massiva  
✅ Schema-on-read (esquema aplicado na leitura)

## ❗ Problemas Comuns em Data Lakes Tradicionais

- ❌ Dificuldade para garantir **confiabilidade**
- ❌ Falta de suporte a **transações ACID**
- ❌ Metadados difíceis de escalar
- ❌ Problemas de leitura durante escrita (writes simultâneos)
- ❌ Linhagem e auditoria limitadas
- ❌ Dificuldade em garantir qualidade dos dados
- ❌ Sem suporte a operações UPDATE/DELETE
- ❌ Problemas de consistência com escritas concorrentes

# Delta Lake - A Camada de Confiabilidade

Delta Lake é a camada de armazenamento otimizada que fornece a fundação para tabelas em um lakehouse no Databricks. É um software open-source que estende arquivos de dados Parquet com um log de transações baseado em arquivo para transações ACID e tratamento escalável de metadados. 

## ✅ O que Delta Lake **É**

- ✅ Tecnologia **open-source** (Linux Foundation)
- ✅ Um **framework de armazenamento** que adiciona confiabilidade ao Data Lake
- ✅ Fundamento da arquitetura **Lakehouse**
- ✅ Compatível com Apache Spark APIs
- ✅ Desenvolvido para integração estreita com Structured Streaming, permitindo usar uma única cópia de dados para operações batch e streaming 

## ❌ O que Delta Lake **Não É**

- ❌ Não é tecnologia proprietária
- ❌ Não é formato de armazenamento (usa Parquet)
- ❌ Não é um banco de dados ou data warehouse
- ❌ Não é um sistema de mensageria

## 🎯 Por que Delta Lake é Necessário?

Delta Lake é o formato padrão para todas as operações no Azure Databricks. A menos que especificado de outra forma, todas as tabelas no Azure Databricks são tabelas Delta. 

**Delta Lake resolve:**
1. Falta de transações ACID em object storage
2. Dificuldade em UPDATE/DELETE em data lakes
3. Problemas de concorrência (múltiplas escritas simultâneas)
4. Falta de versionamento e time travel
5. Schema enforcement
6. Metadados escaláveis

---

# Estrutura de uma Tabela Delta

Uma tabela Delta possui dois componentes principais:

## 1️⃣ Arquivos de Dados (Parquet)

- Armazenam os dados reais
- Formato colunar otimizado
- Compressão eficiente
- Particionamento opcional

## 2️⃣ Transaction Log (`_delta_log/`)

- Diretório contendo arquivos JSON
- Registra todas as transações
- Single Source of Truth

### 📁 Estrutura no Storage
```
/path/to/delta-table/
├── part-00000-xxx.parquet
├── part-00001-xxx.parquet
├── part-00002-xxx.parquet
└── _delta_log/
    ├── 00000000000000000000.json  ← Transação 0
    ├── 00000000000000000001.json  ← Transação 1
    ├── 00000000000000000002.json  ← Transação 2
    └── 00000000000000000010.checkpoint.parquet  ← Checkpoint
```

### 📸 Diagrama Ilustrativo

![delta log structure](https://www.databricks.com/sites/default/files/styles/max_1000x1000/public/2025-04/diving-into-delta-lake-unpacking-the-transaction-log-2x.png?itok=WS_HJX9-&v=1743579811)

---

# Transaction Log - O Coração do Delta Lake

O Delta Lake transaction log (também conhecido como DeltaLog) é um registro ordenado de todas as transações que já foram realizadas em uma tabela Delta Lake desde sua criação. 

## 📋 O que o Transaction Log Contém

Cada arquivo JSON no `_delta_log` contém:

- **Operação realizada**: INSERT, UPDATE, DELETE, MERGE, OPTIMIZE
- **Arquivos adicionados**: Novos arquivos Parquet criados
- **Arquivos removidos**: Arquivos Parquet marcados como inválidos
- **Predicados utilizados**: Condições WHERE aplicadas
- **Metadados**: Estatísticas, schema, partições
- **Timestamp**: Quando a operação ocorreu
- **Informações de commit**: Quem fez a operação

## 🎯 Benefícios do Transaction Log

✅ **Single Source of Truth**: O log de transações serve como fonte única da verdade - o repositório central que rastreia todas as mudanças que usuários fazem na tabela 

✅ **Auditoria Completa**: Trilha completa de todas as mudanças (audit trail)

✅ **Isolamento**: Permite leituras e escritas simultâneas sem conflitos

✅ **Time Travel**: Podemos recriar o estado de uma tabela em qualquer ponto no tempo começando com uma tabela original e processando apenas commits feitos antes daquele ponto 

✅ **ACID Compliance**: Garante transações atômicas, consistentes, isoladas e duráveis

O log de transações Delta Lake tem um protocolo aberto bem definido que pode ser usado por qualquer sistema para ler o log. 

---

# Como Funcionam Writes e Reads

## 1️⃣ Primeira Escrita (Initial Write)
```python
# Processo de escrita
df.write.format("delta").save("/path/to/table")
```

**O que acontece:**
1. ✍️ Escrever arquivos Parquet no diretório da tabela
2. 📝 Criar entrada no transaction log (`000.json`)
3. ✅ Commit bem-sucedido - dados visíveis

**Estado do Storage:**
```
Data Files:
├── part-00000-xxx.parquet  ✓
├── part-00001-xxx.parquet  ✓

_delta_log:
└── 00000000000000000000.json  ✓
```

## 2️⃣ Leitura (Read)
```python
# Processo de leitura
df = spark.read.format("delta").load("/path/to/table")
```

**O que acontece:**
1. 📖 Ler transaction log para descobrir versão atual
2. 📋 Identificar quais arquivos Parquet são válidos
3. 📊 Ler apenas os arquivos Parquet válidos

Quando um usuário lê uma tabela Delta Lake pela primeira vez ou executa uma nova query em uma tabela aberta que foi modificada desde a última leitura, o Spark verifica o transaction log para ver quais novas transações foram postadas na tabela, e então atualiza a versão do usuário da tabela com essas novas mudanças. 

## 3️⃣ Atualizações (Updates)
```sql
UPDATE table_name
SET column = new_value
WHERE condition
```

**O que acontece (Copy-on-Write):**
1. 📖 Ler transaction log para identificar arquivos afetados
2. 📄 Ler arquivos Parquet que contêm dados a atualizar
3. ✍️ Criar **novos** arquivos Parquet com dados atualizados
4. 🗑️ Marcar arquivos antigos como removidos no log
5. ✅ Adicionar arquivos novos no log
6. 📝 Commit da transação

**IMPORTANTE:** Arquivos antigos **NÃO são deletados fisicamente**, apenas marcados como inválidos no transaction log.

**Estado após Update:**
```
Data Files:
├── part-00000-xxx.parquet  ⊘ (marcado como removido no log)
├── part-00001-xxx.parquet  ✓ (ainda válido)
├── part-00002-xxx.parquet  ✓ (novo arquivo com updates)

_delta_log:
├── 00000000000000000000.json
└── 00000000000000000001.json  ✓ (registra update)
```

## 4️⃣ Escritas e Leituras Simultâneas

Delta Lake usa controle de concorrência otimista para fornecer garantias transacionais entre escritas. 

**Cenário:**
- Writer Process está escrevendo novos dados
- Reader Process está lendo dados existentes

**O que acontece:**
```
Timeline:
|
├─ [Writer] Escreve part-00003.parquet
├─ [Reader] Lê transaction log (até versão 1)
├─ [Reader] Lê part-00001.parquet e part-00002.parquet
├─ [Writer] Adiciona 00000000000000000002.json
└─ [Reader] Próxima leitura verá a nova versão
```

É incrivelmente rápido porque ao lidar com petabytes de dados, há grande probabilidade de que usuários trabalhem em partes diferentes dos dados, permitindo completar transações não-conflitantes simultaneamente. 

## 5️⃣ Falhas na Escrita (Failed Writes)
```python
# Simulando falha
try:
    df.write.format("delta").save("/path/to/table")
    # [FALHA OCORRE AQUI]
except Exception as e:
    print("Write failed!")
```

**O que acontece:**
1. ✍️ Arquivos Parquet podem ser escritos no storage
2. ❌ Falha ocorre antes do commit no transaction log
3. 📋 Transaction log **NÃO é atualizado**
4. ✅ Leitores continuam vendo apenas dados válidos

Arquivos de dados não são rastreados a menos que o transaction log registre uma nova versão. Se uma transação falhar após escrever arquivos de dados em uma tabela, esses arquivos não corromperão o estado da tabela, mas os arquivos não se tornarão parte da tabela. 

**Limpeza:**
A operação VACUUM deleta todos os arquivos de dados não rastreados no diretório da tabela, incluindo arquivos não commitados de transações que falharam. 
```sql
-- Remove arquivos órfãos
VACUUM table_name RETAIN 168 HOURS
```

---

# Garantias ACID

Azure Databricks usa Delta Lake por padrão para todas as leituras e escritas e constrói sobre as garantias ACID fornecidas pelo protocolo open-source Delta Lake. 

## A - Atomicidade (Atomicity)

Atomicidade significa que todas as transações ou completam totalmente ou falham completamente. 

**Exemplo:**
```python
# Escrevendo 100 arquivos
# Se falhar no arquivo 40:
# - Data Lake tradicional: 40 arquivos são adicionados (parcial)
# - Delta Lake: NENHUM arquivo é adicionado (tudo ou nada)
```

O transaction log controla a atomicidade dos commits. Durante uma transação, arquivos de dados são escritos no diretório de arquivos que suporta a tabela. Quando a transação completa, uma nova entrada é commitada no transaction log que inclui os caminhos para todos os arquivos escritos durante a transação. 

## C - Consistência (Consistency)

Garantias de consistência se relacionam a como um dado estado dos dados é observado por operações simultâneas. 

**Schema Enforcement:**
- Delta Lake valida schema na escrita
- Garante que todos os dados escritos correspondem aos requisitos
- Previne dados corrompidos ou incompatíveis

## I - Isolamento (Isolation)

Isolamento refere-se a como operações simultâneas potencialmente conflitam umas com as outras. 

**Níveis de Isolamento:**
Azure Databricks usa isolamento write serializable por padrão para todas as escritas e atualizações de tabela. Isolamento de snapshot é usado para todas as leituras de tabela. 

**Controle de Concorrência Otimista:**
Delta Lake usa controle de concorrência otimista para fornecer garantias transacionais entre escritas. Sob este mecanismo, escritas operam em três estágios: 

1. **Read**: Lê a versão mais recente da tabela para identificar quais arquivos precisam ser modificados
2. **Write**: Escreve arquivos de dados no diretório usado para definir a tabela
3. **Validate & Commit**: Verifica se as mudanças propostas conflitam com outras mudanças commitadas desde o snapshot lido

## D - Durabilidade (Durability)

Durabilidade significa que mudanças commitadas são permanentes. 

Azure Databricks usa cloud object storage para armazenar todos os arquivos de dados e transaction logs. Cloud object storage tem alta disponibilidade e durabilidade. Como transações ou completam totalmente ou falham completamente e o transaction log vive junto com arquivos de dados no cloud object storage, tabelas no Azure Databricks herdam as garantias de durabilidade do cloud object storage onde estão armazenadas. 

Referências:
- https://www.databricks.com/blog/2019/08/21/diving-into-delta-lake-unpacking-the-transaction-log.html
- https://learn.microsoft.com/en-us/azure/databricks/delta/
- https://learn.microsoft.com/en-us/azure/databricks/lakehouse/acid
- https://learn.microsoft.com/en-us/azure/databricks/delta/tutorial